[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/laboratorios/S12/S12_07_autoencoder_generativos.ipynb)

# S12 · 07 · Autoencoders y modelos generativos (GAN y difusión)

**Tiempo estimado:** 50 min · **Sprint 12** (UD 4.10) · Cliente: **TurisData Canarias**

**Lo que vas a aprender**
1. Qué es un **autoencoder**: una red que aprende a comprimir y reconstruir.
2. Usarlo para **detectar anomalías** en la ocupación (días "raros" = mal reconstruidos).
3. Explicar con esquemas la idea de las **GAN** y de los **modelos de difusión** (sin entrenarlos).
4. Valorar **riesgos** de la IA generativa (contenido falso, derechos, sesgos).

**Contexto.** A TurisData le interesa detectar **semanas atípicas** de ocupación (un evento, un fallo del sistema de reservas, una campaña)
para investigarlas. No tenemos etiquetas de "anómalo": el autoencoder aprende lo **normal** y señala lo que no encaja.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import os, random, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore")
SEMILLA = 42
random.seed(SEMILLA); np.random.seed(SEMILLA); keras.utils.set_random_seed(SEMILLA)
plt.rcParams.update({"figure.figsize": (10, 3.5), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

ocup = pd.read_csv(dato("ocupacion_diaria.csv"), parse_dates=["fecha"]).set_index("fecha")

## 1. Idea: comprimir y reconstruir
Un **autoencoder** tiene dos mitades:
- **Codificador:** reduce la entrada a pocos números (el **cuello de botella**, "resumen").
- **Decodificador:** intenta reconstruir la entrada original a partir de ese resumen.

Se entrena para que la salida sea **igual a la entrada**. Como el cuello es estrecho, la red solo puede guardar **lo esencial**:
los patrones habituales. Cuando le llega algo raro, lo reconstruye mal → **error de reconstrucción alto = posible anomalía**.

Vamos a representar cada **semana** (7 días de ocupación) como un vector de 7 números.

In [ ]:
serie = ocup["ocupacion_pct"]
n_semanas = len(serie) // 7
semanas = serie.values[:n_semanas * 7].reshape(n_semanas, 7)
fechas_semana = serie.index[:n_semanas * 7:7]
print("Semanas:", semanas.shape, "| primera semana empieza el", fechas_semana[0].date())

**Ejercicio 1.** Reserva el **último 20 %** de las semanas para prueba (orden temporal, sin barajar) y **escala** con `MinMaxScaler`
ajustado solo con las semanas de entrenamiento. Ya tienes `corte` (dónde acaba el entrenamiento). Guarda el escalador en `escalador` y las matrices ya escaladas (7 columnas) en `sem_ent` y `sem_test`.

In [ ]:
corte = int(0.8 * n_semanas)      # posición donde termina el entrenamiento y empieza la prueba
# TODO: ajusta el escalador solo con las semanas anteriores a corte y transforma después los dos trozos
...
print("Entrenamiento:", sem_ent.shape, "| prueba:", sem_test.shape)
assert sem_ent.shape[1] == 7 and sem_test.shape[1] == 7, "Cada fila es una semana de 7 valores."
assert abs(sem_ent.min()) < 1e-9 and abs(sem_ent.max() - 1) < 1e-9, "El escalador se ajusta solo con entrenamiento: min 0 y max 1 en sem_ent."
assert len(sem_ent) + len(sem_test) == n_semanas, "No pierdas semanas al partir."

## 2. Construir y entrenar el autoencoder
Arquitectura pequeña: 7 → 4 → **2** (cuello) → 4 → 7. Con solo 2 números tiene que resumir cada semana.

In [ ]:
def construir_autoencoder(cuello=2):
    entrada = keras.Input(shape=(7,))
    x = layers.Dense(4, activation="relu")(entrada)
    z = layers.Dense(cuello, activation="relu", name="cuello")(x)
    x = layers.Dense(4, activation="relu")(z)
    salida = layers.Dense(7, activation="sigmoid")(x)
    modelo = keras.Model(entrada, salida)
    modelo.compile(optimizer=keras.optimizers.Adam(0.01), loss="mse")
    return modelo

keras.utils.set_random_seed(SEMILLA)
ae = construir_autoencoder()
ae.summary(print_fn=lambda l: print(l))

**Ejercicio 2.** Entrena `ae` con `sem_ent` como **entrada y objetivo** (¡el mismo array dos veces!), 150 épocas, `batch_size=16`, `verbose=0`,
con el 15 % final como validación (`validation_split=0.15`). Guarda el historial en `historial`.

In [ ]:
# TODO: ae.fit(sem_ent, sem_ent, epochs=150, batch_size=16, validation_split=0.15, verbose=0)
...
print("Error final de entrenamiento:", round(historial.history["loss"][-1], 4), "| validación:", round(historial.history["val_loss"][-1], 4))
assert len(historial.history["loss"]) == 150, "Entrena 150 épocas."
assert historial.history["loss"][-1] < historial.history["loss"][0], "El error debería bajar con el entrenamiento; recuerda usar sem_ent como entrada Y como objetivo."

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(historial.history["loss"], label="entrenamiento"); ax.plot(historial.history["val_loss"], label="validación")
ax.set_title("Error de reconstrucción durante el entrenamiento"); ax.set_xlabel("Época"); ax.set_ylabel("Error cuadrático medio"); ax.legend()
plt.show()

## 3. Ver una reconstrucción
Comparamos una semana real con la que reconstruye el autoencoder (deshaciendo el escalado).

In [ ]:
i = 5
original = sem_test[i:i + 1]
recons = ae.predict(original, verbose=0)
dias = ["lun", "mar", "mié", "jue", "vie", "sáb", "dom"]
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(dias, escalador.inverse_transform(original)[0], "o-", label="Real")
ax.plot(dias, escalador.inverse_transform(recons)[0], "s--", label="Reconstruida")
ax.set_ylabel("Ocupación (%)"); ax.set_title("Una semana de prueba"); ax.legend()
plt.show()

## 4. Detección de anomalías
1. Calculamos el **error de reconstrucción** de cada semana (error cuadrático medio de sus 7 días).
2. Fijamos un **umbral** con las semanas de **entrenamiento** (normales): por ejemplo, el **percentil 95** de sus errores.
3. Una semana con error por encima del umbral se marca como sospechosa.

**Ejercicio 3.** Completa `error_recons(modelo, X)`: devuelve un array con el error cuadrático medio de cada fila de `X`
(media de `(X - modelo.predict(X))**2` por filas). Calcula `umbral` (percentil 95 de los errores de `sem_ent`, `np.percentile`).

In [ ]:
def error_recons(modelo, X):
    # TODO: predice con el modelo y haz la media por filas del cuadrado de la diferencia (axis=1)
    ...

err_ent, err_test = error_recons(ae, sem_ent), error_recons(ae, sem_test)
umbral = float(np.percentile(err_ent, 95))
print(f"Umbral (percentil 95 en entrenamiento): {umbral:.4f}")
assert err_ent.shape == (len(sem_ent),), "error_recons debe devolver un error por semana (media por filas: axis=1)."
assert err_ent.min() >= 0, "Los errores cuadráticos no pueden ser negativos."
assert abs(np.mean(err_ent > umbral) - 0.05) < 0.03, "El umbral debe dejar ~5 % de las semanas de entrenamiento por encima."

### Probamos con anomalías inyectadas
No sabemos qué semanas reales son "raras", así que **fabricamos** casos de prueba para ver si el modelo los detecta:
- Un **apagón de reservas**: una semana con ocupación baja todos los días (p. ej. 15 %).
- Un **día roto**: una semana normal pero con un día a 0 % (fallo del sistema).
- Un **pico irreal**: un día al 100 % en una semana floja.
Se escala igual que el resto de los datos.

In [ ]:
normal = semanas[corte:corte + 3].copy()
raras = np.array([np.full(7, 15.0),                              # apagón
                  normal[0].copy(),                               # día roto
                  np.array([55., 52., 54., 53., 100., 55., 53.])])  # pico irreal
raras[1, 3] = 0.0
raras_esc = escalador.transform(raras)
err_raras = error_recons(ae, raras_esc)
for nombre, e in zip(["Apagón", "Día roto", "Pico irreal"], err_raras):
    print(f"{nombre:12s} error = {e:.4f}  →  {'ANOMALÍA' if e > umbral else 'normal'}")

**Ejercicio 4.** Calcula `n_detectadas` (cuántas de las 3 anomalías inyectadas superan el umbral) y `falsos_positivos`
(cuántas semanas **reales de prueba**, `err_test`, superan el umbral).

In [ ]:
# TODO: suma de booleanos (err > umbral) en cada array
...
print(f"Anomalías detectadas: {n_detectadas}/3 | semanas reales marcadas: {falsos_positivos}/{len(err_test)}")
assert 0 <= n_detectadas <= 3, "n_detectadas es un número entre 0 y 3."
assert n_detectadas >= 2, "El autoencoder debería detectar al menos 2 de las 3 anomalías inyectadas; revisa el umbral y el escalado."
assert falsos_positivos < len(err_test), "No todas las semanas reales pueden ser anómalas."

In [ ]:
fig, ax = plt.subplots()
ax.plot(fechas_semana[corte:], err_test, "o-", ms=3, label="Semanas reales (prueba)")
ax.axhline(umbral, color="red", ls="--", label="Umbral (p95 entrenamiento)")
ax.set_title("Error de reconstrucción por semana (prueba)"); ax.set_ylabel("Error"); ax.set_xlabel("Fecha"); ax.legend()
plt.show()

## 5. Mini-reto integrador: ¿qué semanas reales revisar?
Construye `revisar`: un DataFrame con las semanas de **prueba** cuyo error supera el umbral, con columnas `inicio_semana` (fecha) y `error`,
ordenado de mayor a menor error. Si no hay ninguna, queda vacío (es una buena noticia). Escribe en `interpretacion` (texto)
una frase para el cliente: qué significa que una semana aparezca ahí y qué **no** significa (no es prueba de fallo, solo "merece mirarse").

In [ ]:
# TODO: DataFrame con fechas_semana[corte:] y err_test; filtra error > umbral; sort_values descendente
...
print(revisar.head())
assert list(revisar.columns) == ["inicio_semana", "error"], "Columnas: inicio_semana y error."
assert (revisar["error"] > umbral).all(), "Solo semanas con error por encima del umbral."
assert revisar["error"].is_monotonic_decreasing, "Ordena de mayor a menor error."
assert isinstance(interpretacion, str) and len(interpretacion) > 40, "Escribe una interpretación en una o dos frases."

## 6. Otros modelos generativos: ideas (sin entrenar)
El autoencoder **comprime y reconstruye**. Los modelos **generativos** dan un paso más: **crean datos nuevos** parecidos a los de entrenamiento.
No los entrenamos aquí (necesitan GPU y mucho tiempo), pero conviene entender la idea.

### GAN (*Generative Adversarial Network*): el falsificador y el policía
```
  ruido aleatorio ──► [ GENERADOR ] ──► imagen falsa ─┐
                                                       ├──► [ DISCRIMINADOR ] ──► ¿real o falsa?
                         imágenes reales ─────────────┘            │
       ▲                                                             │
       └───────────── ambos se corrigen con el resultado ◄──────────┘
```
- El **generador** intenta fabricar ejemplos tan buenos que engañen al discriminador.
- El **discriminador** intenta distinguir lo real de lo falso.
- Al competir, ambos mejoran. Resultado: imágenes muy realistas (fotos de habitaciones, caras que no existen).
- **Problemas típicos:** entrenamiento inestable; el generador repite pocos ejemplos (*mode collapse*).

### Modelos de difusión: de la niebla a la imagen
```
  ENTRENAMIENTO (se aprende a quitar ruido):
  imagen ──► +ruido ──► +ruido ──► ... ──► puro ruido       (proceso "hacia delante", fijo)

  GENERACIÓN (se aplica al revés):
  puro ruido ──► red quita un poco ──► ... ──► imagen nueva  (guiada por el texto: "una terraza al atardecer en Tenerife")
```
- Se entrena una red para **quitar un poquito de ruido** en cada paso.
- Para crear, se parte de ruido puro y se limpia paso a paso, guiando el resultado con un texto.
- Estables de entrenar y de gran calidad; son la base de muchos generadores de imágenes actuales. Más lentos que una GAN al generar.

### Comparación rápida
| | Autoencoder | GAN | Difusión |
|---|---|---|---|
| Idea | comprimir y reconstruir | competir (falsificador vs. policía) | quitar ruido paso a paso |
| Uso típico | anomalías, compresión | imágenes realistas | imágenes y vídeo guiados por texto |
| Ventaja | simple, rápido | resultados nítidos | calidad y estabilidad |
| Inconveniente | no genera cosas nuevas por sí solo | inestable | lento al generar, caro |

**Ejercicio 5 (de comprensión).** Asigna a `parejas` un diccionario que empareje cada situación con la técnica **más adecuada**
(`"autoencoder"`, `"gan"` o `"difusion"`):
- `"detectar semanas raras de ocupación"`
- `"generar la foto de una terraza a partir de un texto"`
- `"aprender a engañar a un discriminador con imágenes falsas"`

In [ ]:
# TODO: piensa: sin etiquetas y reconstruyendo → autoencoder; imagen desde texto → difusión; competición generador/discriminador → GAN
...
assert parejas.get("detectar semanas raras de ocupación") == "autoencoder", "Detectar rarezas sin etiquetas: autoencoder."
assert parejas.get("generar la foto de una terraza a partir de un texto") == "difusion", "Imagen desde texto: difusión."
assert parejas.get("aprender a engañar a un discriminador con imágenes falsas") == "gan", "Generador contra discriminador: GAN."

## 7. IA generativa responsable
Si TurisData usara imágenes o textos generados en su web o sus anuncios:
- **Veracidad:** una foto sintética de una habitación que no existe puede ser **publicidad engañosa**. Hay que etiquetar el contenido generado y no usarlo para mostrar el alojamiento real.
- **Derechos:** ¿con qué datos se entrenó el modelo? ¿Quién es el autor de lo generado? Revisa licencias.
- **Sesgos:** los modelos reproducen estereotipos de sus datos (¿cómo se representa al "turista" o al "personal"?).
- **Datos personales:** no subir fotos de huéspedes ni reseñas con nombres a servicios externos.
- **Supervisión humana:** una persona revisa antes de publicar.

## Preguntas de reflexión
1. ¿Por qué el cuello de botella tiene que ser **estrecho**? ¿Qué pasaría con un cuello de 7 neuronas (igual que la entrada)?
2. El umbral del percentil 95 marca ~5 % de las semanas normales como anómalas. ¿Cómo lo cambiarías si revisar una alerta cuesta mucho tiempo?
3. ¿Qué otras variables (temperatura, festivos, precio) podrías incluir en el vector de cada semana?
4. Escribe una norma de uso de imágenes generadas por IA para el equipo de marketing de TurisData (3 líneas).

## Resumen: qué has aprendido
- Un **autoencoder** aprende a comprimir y reconstruir lo normal; el **error de reconstrucción** señala lo raro (anomalías sin etiquetas).
- El umbral se fija con datos normales de **entrenamiento** y se prueba con anomalías inyectadas y semanas reales.
- **GAN** = competición entre generador y discriminador; **difusión** = quitar ruido paso a paso, guiado por texto.
- La IA generativa exige criterios de veracidad, derechos, sesgos y supervisión humana.